# Reported Configuration — save predictions

**Standalone.** Does not depend on `05_skeleton_model.ipynb` still being in
memory; it reproduces the setup cells it needs.

### Why this exists

`stage5_predictions.csv` was written *before* the auxiliary weight sweep, so it
holds the heavy configuration (total aux weight 1.1, macro-F1 0.764) — the
setting the sweep then showed was worst.

The configuration actually reported, `technique_only` at 0.20 (macro-F1 0.792),
produced a single summary number and nothing else: no per-class breakdown, no
confusion matrix, no confidence intervals, no per-technique table. Its
predictions were never saved.

This re-runs 7-fold LOVO with that configuration and writes the out-of-fold
predictions, so every table in the paper recomputes against the model being
reported.

Same training loop as before. Nothing new is learned; the outputs are kept
this time.

**GPU required, ~25 min.**


## 1 · Mount

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2 · Config and data

In [2]:
BASE = "/content/drive/MyDrive/tt_coach"

EPOCHS      = 60
BATCH       = 64
LR          = 2e-3
SEED        = 42
USE_OPPONENT = False      # D8 kept the data; this is an ablation switch

import json, math, random, warnings
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
warnings.filterwarnings("ignore")

BASE = Path(BASE); META = BASE/"derived/meta"; CLIP = BASE/"derived/clips"
OUT  = BASE/"outputs"; (OUT/"metrics").mkdir(parents=True, exist_ok=True)
(OUT/"figures").mkdir(parents=True, exist_ok=True)
(BASE/"models/checkpoints").mkdir(parents=True, exist_ok=True)

dev = "cuda" if torch.cuda.is_available() else "cpu"
assert dev == "cuda", "No GPU. Runtime > Change runtime type > T4."
print(f"device: {torch.cuda.get_device_name(0)}")

def seed_all(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(SEED)

# taxonomy: prefer v2, fall back to v1
import yaml
tax_path = next((p for p in [BASE/"TAXONOMY_v2.yaml", BASE/"TAXONOMY.yaml"]
                 if p.exists()), None)
TAX = yaml.safe_load(tax_path.read_text())
print(f"taxonomy: {tax_path.name} (v{TAX.get('version')})")

d = np.load(CLIP/"canonical.npz", allow_pickle=True)
PRE, NF = int(d["contact_index"]), int(d["n_frames"])
usable  = d["usable"]

CLASSES   = ["serve", "attack", "control", "defence"]
TECHS     = ["block","chop","flick","lob","loop","push","serve","smash"]
LEANS     = ["neutral","back_heavy","front_heavy","right_leaning","left_leaning","unknown"]
FEETS     = ["both_feet_planted","both_feet_lifted","left_foot_lifted","right_foot_lifted","unknown"]

meta = pd.DataFrame({k: d[k] for k in
        ["stroke_id","video_id","fold","shot_class","high_level","technique",
         "lean","feet","eff_hand"]})
print(f"{usable.sum()} usable of {len(meta)}")

device: Tesla T4
taxonomy: TAXONOMY_v2.yaml (v2)
1432 usable of 1457


## 3 · Tensors

In [3]:
KP   = d["kp"].astype(np.float32)      # (N,2,T,17,2)
VEL  = d["vel"].astype(np.float32)
VAL  = d["valid"].astype(np.float32)
TD   = np.nan_to_num(d["table_dist"].astype(np.float32), nan=0.0)

def build(idx, players=(0,)):
    xs = []
    for p in players:
        kp  = KP[idx][:, p].reshape(len(idx), NF, -1)     # 34
        vel = VEL[idx][:, p].reshape(len(idx), NF, -1)    # 34
        val = VAL[idx][:, p]                              # 17
        xs += [kp, vel, val]
    xs.append(TD[idx][..., None])                         # 1
    x = np.concatenate(xs, -1)                            # (N,T,C)
    return np.nan_to_num(x).transpose(0, 2, 1)            # (N,C,T)

ALL = np.arange(len(meta))
PLAYERS = (0, 1) if USE_OPPONENT else (0,)
X = build(ALL, PLAYERS)
C_IN = X.shape[1]
print(f"X {X.shape}  ({C_IN} channels x {NF} frames)")

def enc(col, vocab):
    m = {v: i for i, v in enumerate(vocab)}
    return meta[col].map(m).fillna(-1).astype(int).values

Y = {
    "shot":  enc("shot_class", CLASSES),
    "tech":  enc("technique",  TECHS),
    "fhbh":  enc("high_level", ["forehand","backhand"]),
    "lean":  enc("lean",  LEANS),
    "feet":  enc("feet",  FEETS),
}
N_OUT = {"shot":4, "tech":8, "fhbh":2, "lean":len(LEANS), "feet":len(FEETS)}
W_AUX = {"tech":0.3, "fhbh":0.3, "lean":0.3, "feet":0.2}

for k, v in Y.items():
    print(f"  {k:<5} {N_OUT[k]} classes, {(v<0).sum()} missing")

X (1457, 86, 97)  (86 channels x 97 frames)
  shot  4 classes, 0 missing
  tech  8 classes, 0 missing
  fhbh  2 classes, 0 missing
  lean  6 classes, 0 missing
  feet  5 classes, 1 missing


## 4 · Model

In [4]:
class Block(nn.Module):
    def __init__(s, c, dil, drop=0.2):
        super().__init__()
        pad = dil * 2
        s.c1 = nn.Conv1d(c, c, 5, padding=pad, dilation=dil)
        s.c2 = nn.Conv1d(c, c, 5, padding=pad, dilation=dil)
        s.n1, s.n2 = nn.BatchNorm1d(c), nn.BatchNorm1d(c)
        s.do = nn.Dropout(drop)
    def forward(s, x):
        r = x
        x = s.do(F.gelu(s.n1(s.c1(x))))
        x = s.do(F.gelu(s.n2(s.c2(x))))
        return F.gelu(x + r)


class AttnPool(nn.Module):
    def __init__(s, c):
        super().__init__(); s.score = nn.Conv1d(c, 1, 1)
    def forward(s, x):                       # (B,C,T)
        w = torch.softmax(s.score(x), -1)
        return torch.cat([(x * w).sum(-1), x.max(-1).values], -1)


class Net(nn.Module):
    def __init__(s, c_in, width=128, multitask=True):
        super().__init__()
        s.multitask = multitask
        s.stem = nn.Sequential(nn.Conv1d(c_in, width, 1),
                               nn.BatchNorm1d(width), nn.GELU())
        s.blocks = nn.Sequential(*[Block(width, dl) for dl in (1,2,4,8,16,32)])
        s.pool = AttnPool(width)
        s.trunk = nn.Sequential(nn.Linear(width*2, 256), nn.GELU(), nn.Dropout(0.3))
        heads = ["shot"] + (list(W_AUX) if multitask else [])
        s.heads = nn.ModuleDict({h: nn.Linear(256, N_OUT[h]) for h in heads})
    def forward(s, x):
        z = s.trunk(s.pool(s.blocks(s.stem(x))))
        return {h: s.heads[h](z) for h in s.heads}


def focal(logits, target, weight=None, gamma=2.0):
    m = target >= 0
    if m.sum() == 0:
        return logits.sum() * 0.0
    logits, target = logits[m], target[m]
    ce = F.cross_entropy(logits, target, weight=weight, reduction="none")
    pt = torch.exp(-F.cross_entropy(logits, target, reduction="none"))
    return ((1 - pt) ** gamma * ce).mean()

print(f"params: {sum(p.numel() for p in Net(C_IN).parameters()):,}")

params: 1,071,386


## 5 · Augmentation

In [5]:
def augment(x):
    B, C, T = x.shape
    # contact jitter +/- 6 frames
    sh = torch.randint(-6, 7, (B,), device=x.device)
    idx = (torch.arange(T, device=x.device)[None] + sh[:, None]).clamp(0, T-1)
    x = torch.gather(x, 2, idx[:, None].expand(-1, C, -1))
    # temporal speed warp 0.85 - 1.15x
    if random.random() < 0.5:
        sc = random.uniform(0.85, 1.15)
        x = F.interpolate(x, size=max(8, int(T*sc)), mode="linear",
                          align_corners=False)
        x = F.interpolate(x, size=T, mode="linear", align_corners=False)
    # keypoint dropout: zero whole channels
    if random.random() < 0.3:
        x = x * (torch.rand(B, C, 1, device=x.device) > 0.1).float()
    # gaussian jitter
    x = x + torch.randn_like(x) * 0.01
    return x

print("augmentation ready")

augmentation ready


## 6 · Training loop

In [6]:
from sklearn.metrics import f1_score

def run_fold(f, multitask, Xa, verbose=False):
    tr = ((meta.fold != f) & usable).values
    va = ((meta.fold == f) & usable).values

    xt = torch.tensor(Xa[tr], device=dev)
    xv = torch.tensor(Xa[va], device=dev)
    mu, sd = xt.mean((0,2), keepdim=True), xt.std((0,2), keepdim=True) + 1e-6
    xt, xv = (xt-mu)/sd, (xv-mu)/sd

    yt = {k: torch.tensor(v[tr], device=dev) for k, v in Y.items()}
    yv = {k: torch.tensor(v[va], device=dev) for k, v in Y.items()}

    cnt = np.bincount(Y["shot"][tr], minlength=4).clip(1)
    cw = torch.tensor(len(Y["shot"][tr])/(4*cnt), dtype=torch.float32, device=dev)

    net = Net(Xa.shape[1], multitask=multitask).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.OneCycleLR(
        opt, LR, total_steps=EPOCHS*max(1, math.ceil(tr.sum()/BATCH)))

    # balanced sampling so defence is not buried by the 2.9:1 imbalance
    p = (1.0/cnt)[Y["shot"][tr]]; p = p/p.sum()
    best, best_state = -1, None

    for ep in range(EPOCHS):
        net.train()
        order = np.random.choice(tr.sum(), tr.sum(), p=p)
        for i in range(0, len(order), BATCH):
            b = order[i:i+BATCH]
            if len(b) < 4: continue
            xb = augment(xt[b])
            out = net(xb)
            loss = focal(out["shot"], yt["shot"][b], cw)
            if multitask:
                for h, w in W_AUX.items():
                    loss = loss + w * focal(out[h], yt[h][b])
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step(); sch.step()

        if ep >= 15 and ep % 3 == 0:
            net.eval()
            with torch.no_grad():
                pv = net(xv)["shot"].argmax(1).cpu().numpy()
            s = f1_score(Y["shot"][va], pv, average="macro", zero_division=0)
            if s > best:
                best = s
                best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}

    if best_state is not None:            # EPOCHS <= 15 would leave it unset
        net.load_state_dict(best_state)
    net.eval()
    with torch.no_grad():
        logits = net(xv)["shot"]
        prob = torch.softmax(logits, 1).cpu().numpy()
    return prob, va, best

print("training loop ready")

training loop ready


## 7 · Run and save

`W_AUX` is set to `technique_only` before training, so `Net` builds the right
heads and the loss uses the right weight.

In [7]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

REPORTED_AUX = {"tech": 0.20}
OUT_NAME = "stage5_technique_only_predictions.csv"

globals()["W_AUX"] = REPORTED_AUX       # Net reads list(W_AUX) for its heads
seed_all(SEED)

FOLDS = sorted(meta.fold[usable].unique())
print(f"configuration : technique_only, total aux weight "
      f"{sum(REPORTED_AUX.values()):.2f}")
print(f"folds         : {FOLDS}\n")

oof = np.zeros((len(meta), 4), np.float32)
covered = np.zeros(len(meta), bool)

for f in FOLDS:
    prob, va, best = run_fold(f, multitask=True, Xa=X)
    oof[va] = prob
    covered |= va
    yt, yp_ = Y["shot"][va], prob.argmax(1)
    print(f"  fold {f}: macro-F1 "
          f"{f1_score(yt, yp_, average='macro', zero_division=0):.3f}"
          f"   acc {(yt == yp_).mean():.3f}   n={int(va.sum())}")

assert covered[usable].all(), "some usable strokes were never predicted"

# --- write in the same shape as stage4/stage5_predictions.csv ---------------
u = usable & covered
pred_idx = oof[u].argmax(1)

out = meta[u].copy()
out["pred"] = [CLASSES[i] for i in pred_idx]
out["conf"] = oof[u].max(1)
out["correct"] = out.pred == out.shot_class
for i, c in enumerate(CLASSES):        # kept for later calibration work
    out[f"p_{c}"] = oof[u][:, i]

cols = [c for c in ("stroke_id","video_id","fold","shot_class",
                    "high_level","technique","eff_hand") if c in out.columns]
out = out[cols + ["pred","conf","correct"] + [f"p_{c}" for c in CLASSES]]

path = OUT/"metrics"/OUT_NAME
out.to_csv(path, index=False)
print(f"\n-> {path}   ({len(out)} strokes)")

configuration : technique_only, total aux weight 0.20
folds         : ['A', 'B', 'C', 'D', 'E', 'F', 'G']

  fold A: macro-F1 0.750   acc 0.731   n=160
  fold B: macro-F1 0.848   acc 0.878   n=385
  fold C: macro-F1 0.795   acc 0.809   n=152
  fold D: macro-F1 0.738   acc 0.729   n=170
  fold E: macro-F1 0.663   acc 0.742   n=248
  fold F: macro-F1 0.886   acc 0.884   n=155
  fold G: macro-F1 0.754   acc 0.784   n=162

-> /content/drive/MyDrive/tt_coach/outputs/metrics/stage5_technique_only_predictions.csv   (1432 strokes)


## 8 · What the paper will report

In [8]:
yt, yp_ = Y["shot"][u], pred_idx

print("=" * 74)
print("REPORTED CONFIGURATION — technique_only @ 0.20")
print("=" * 74)
print(classification_report(yt, yp_, target_names=CLASSES, digits=3,
                            zero_division=0))

cm = confusion_matrix(yt, yp_, labels=range(4))
dfm = pd.DataFrame(cm, index=[f"true_{c}" for c in CLASSES],
                   columns=[f"pred_{c}" for c in CLASSES])
dfm["recall"] = (np.diag(cm)/cm.sum(1)).round(3)
print(dfm.to_string())

macro_new = f1_score(yt, yp_, average="macro", zero_division=0)
print("\n" + "=" * 74)
print(f"  macro-F1 {macro_new:.3f}")
print(f"  aux sweep reported 0.792 for this configuration")
print(f"  previously saved run (heavy aux, 1.1): 0.764")
print(f"  difference vs saved run: {macro_new - 0.764:+.3f}")
print("=" * 74)

drift = abs(macro_new - 0.792)
if drift > 0.010:
    print(f"""
  !! {drift:.3f} away from the sweep's 0.792.

  cuDNN kernel selection is not deterministic by default, so a re-run at a
  fixed seed drifts a little. But if run-to-run variance is this large, it is
  comparable to the differences the sweep was measuring, which would make
  "technique_only is best" a weaker conclusion than it currently looks.

  Worth reporting honestly, and worth a second run to see the spread.""")
else:
    print(f"""
  Within {drift:.3f} of the sweep. Run-to-run variance is small relative to
  the configuration differences, so the sweep's ranking holds.""")

print(f"""
REPORT THIS RUN, not 0.792. The per-class numbers, confusion matrix and
confidence intervals in the paper all come from these predictions, and
quoting a macro-F1 from a different run would be the same inconsistency in
a new place.

NEXT
  In the paper materials notebook, change:

      s5 = load("{OUT_NAME}")

  Table 4 then becomes a real comparison rather than two isolated
  configurations with one mislabelled 'cascade':

      technique_only on ground-truth windows   {macro_new:.3f}
      end-to-end cascade (Stage 7)             0.762""")

REPORTED CONFIGURATION — technique_only @ 0.20
              precision    recall  f1-score   support

       serve      0.989     0.964     0.976       277
      attack      0.857     0.782     0.817       650
     control      0.773     0.857     0.813       279
     defence      0.523     0.602     0.560       226

    accuracy                          0.803      1432
   macro avg      0.786     0.801     0.792      1432
weighted avg      0.813     0.803     0.807      1432

              pred_serve  pred_attack  pred_control  pred_defence  recall
true_serve           267            4             2             4   0.964
true_attack            1          508            41           100   0.782
true_control           2           18           239            20   0.857
true_defence           0           63            27           136   0.602

  macro-F1 0.792
  aux sweep reported 0.792 for this configuration
  previously saved run (heavy aux, 1.1): 0.764
  difference vs saved run: +0.028

---
## Output

`outputs/metrics/stage5_technique_only_predictions.csv`

Columns match `stage4_predictions.csv` so the paper notebook's `yp()`,
`tech_acc()` and paired-bootstrap merge all work unchanged, plus four
probability columns the earlier files lacked.
